# UNHD Part A -- Qwen3-VL-8B-Instruct via vLLM
Same model, same evaluation set, same prompt/decoding policy as the transformers-based notebook -- but using vLLM as the inference engine instead of raw `transformers.generate()`. The plain-transformers version processed images at ~23 sec/image (device_map splitting + per-image sequential generation); vLLM's continuous batching and optimized kernels should be dramatically faster for this exact workload (many short requests to one model).

## Cell 0 -- HF token setup (READ FIRST)
Add your (rotated, if you ever pasted the old one anywhere) token as a Kaggle Secret named `HF_TOKEN` via **Add-ons -> Secrets**, toggled on for this notebook. Never paste it into a cell directly.

## Cell 1 -- Install vLLM
We let vLLM manage its own compatible torch/transformers versions rather than fighting it -- it may upgrade or downgrade what's preinstalled on Kaggle to match what it needs. This is expected and correct.

In [ ]:
!pip install -q -U vllm
!pip install -q -U "bitsandbytes>=0.49.2"
!pip install -q qwen-vl-utils==0.0.14 python-Levenshtein

import ctypes
import glob
import os
import site

# T4 has only ~14.5 GiB usable. The vision encoder's activation memory for a batch of images
# isn't covered by vLLM's KV-cache-based gpu_memory_utilization budget, so batches of many
# images can OOM even with headroom nominally free -- expandable_segments lets PyTorch's
# caching allocator reuse reserved-but-fragmented memory instead of failing a contiguous
# allocation (this is exactly what the OOM error itself suggests trying). Must be set before
# torch initializes its CUDA allocator, so do it before anything imports torch/vllm.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# bitsandbytes' CUDA 13 build needs libnvJitLink.so.13 at runtime, and the search-path fix
# (adding site-packages/nvidia/*/lib to LD_LIBRARY_PATH) does NOT work here: on Linux, ld.so
# parses LD_LIBRARY_PATH once at process startup and caches it -- changing os.environ from
# inside an already-running Python process has no effect on ctypes' dlopen() calls, even when
# vLLM loads the model in-process (confirmed: this still failed with VLLM_ENABLE_V1_MULTIPROCESSING
# disabled). The reliable fix is to skip path search entirely: explicitly load the exact .so by
# full path with RTLD_GLOBAL *before* bitsandbytes needs it. That puts it in the process's global
# symbol table, so bitsandbytes' own native module finds it already resident -- no search needed.
nvjitlink_candidates = sorted(set(
    path
    for sp in site.getsitepackages() + [site.getusersitepackages()]
    for path in glob.glob(os.path.join(sp, "nvidia", "*", "lib", "libnvJitLink.so.13"))
))

preloaded = False
for candidate in nvjitlink_candidates:
    try:
        ctypes.CDLL(candidate, mode=ctypes.RTLD_GLOBAL)
        print(f"Preloaded {candidate} into the process (RTLD_GLOBAL).")
        preloaded = True
        break
    except OSError as e:
        print(f"Could not preload {candidate}: {e}")

if not preloaded:
    print("WARNING: could not find/preload libnvJitLink.so.13 anywhere under site-packages -- "
          "bitsandbytes will likely still fail to load its native library.")

# Also keep LD_LIBRARY_PATH populated as a defensive fallback (harmless even though it won't
# retroactively affect this process; it protects any code path that DOES spawn a fresh subprocess).
nvidia_lib_dirs = set()
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    nvidia_lib_dirs.update(glob.glob(os.path.join(sp, "nvidia", "*", "lib")))
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Keep model loading in-process (no separate EngineCore subprocess), so the preload above is
# guaranteed to be visible to the code that actually needs libnvJitLink.so.13.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import vllm
print("vLLM version:", vllm.__version__)

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets and set as environment variable.")
except Exception as e:
    print("No HF_TOKEN secret found -- proceeding without auth (model is public/ungated).")
    print(f"Error was: {e}")

## Cell 2 -- Locate dataset automatically

In [ ]:
def find_dataset_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "evaluation_set.csv" in files:
            return root
    raise FileNotFoundError("Could not find evaluation_set.csv anywhere under /kaggle/input")

DATASET_ROOT = find_dataset_root()
CSV_PATH = os.path.join(DATASET_ROOT, "evaluation_set.csv")
IMAGES_DIR = os.path.join(DATASET_ROOT, "images")

print("CSV:", CSV_PATH)
print("Images dir:", IMAGES_DIR)
print("Images found:", len(os.listdir(IMAGES_DIR)))

## Cell 3 -- Load evaluation set, sanity-check against images on disk

In [ ]:
import csv

with open(CSV_PATH, encoding="utf-8-sig") as f:
    eval_rows = list(csv.DictReader(f))

print(f"Evaluation set rows: {len(eval_rows)}")

def image_filename(row):
    return row["image_path"].replace("\\", "/").rsplit("/", 1)[-1]

missing = [r for r in eval_rows if not os.path.isfile(os.path.join(IMAGES_DIR, image_filename(r)))]
print(f"Missing images: {len(missing)}")
if missing:
    print("Examples:", [image_filename(r) for r in missing[:5]])
    print("STOP -- investigate before running the full pipeline.")

## Cell 4 -- CER/WER functions (same definitions as the other notebooks, for direct comparability)

In [ ]:
import unicodedata

def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            curr[j] = min(prev[j] + 1, curr[j-1] + 1, prev[j-1] + cost)
        prev = curr
    return prev[m]

def strip_diacritics(text):
    normalized = unicodedata.normalize("NFC", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")

def clean_text(text):
    return unicodedata.normalize("NFC", (text or "").strip())

def cer(pred, gt):
    gt = clean_text(gt); pred = clean_text(pred)
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt):
    gt_words = clean_text(gt).split(); pred_words = clean_text(pred).split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gt_words) / len(gt_words)

def compute_metrics(pred, gt):
    cer_sens = cer(pred, gt)
    wer_sens = wer(pred, gt)
    pred_stripped = strip_diacritics(pred)
    gt_stripped = strip_diacritics(gt)
    cer_insens = cer(pred_stripped, gt_stripped)
    wer_insens = wer(pred_stripped, gt_stripped)
    return cer_sens, cer_insens, wer_sens, wer_insens

assert edit_distance("kitten", "sitting") == 3
print("Metric functions OK")

## Cell 5 -- Load Qwen3-VL-8B-Instruct with vLLM (bitsandbytes 4-bit, single GPU by default)

In [ ]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    dtype="float16",
    max_model_len=4096,          # generous for a single short handwriting line; sanity test in Cell 7 will
                                    # surface an error immediately if this is too small, rather than failing silently
    gpu_memory_utilization=0.75,   # leave more headroom for the vision encoder's activation memory,
                                    # which isn't covered by this KV-cache-based budget (a T4 only
                                    # has ~14.5 GiB usable; 0.85 left too little slack and OOM'd when
                                    # batching many images -- see Cell 8)
    max_num_batched_tokens=2048,   # caps how many images' vision embeddings get computed together in
                                    # one scheduler step, directly bounding the vision-encoder memory
                                    # spike; vLLM's continuous batching still processes the full outer
                                    # batch, just in more (smaller) internal steps -- throughput impact
                                    # should be minor since chunked prefill was already limiting this
    limit_mm_per_prompt={"image": 1},
    trust_remote_code=True,
    # vLLM defaults to a single GPU (tensor_parallel_size=1) unless told otherwise --
    # no explicit device pinning needed, unlike the transformers version's device_map issue.
)

OCR_PROMPT = (
    "This image contains a single line of handwritten Urdu text in the Nastaliq script. "
    "Transcribe exactly what is written, character for character, including any diacritics. "
    "Output ONLY the transcribed Urdu text -- no translation, no explanation, no extra commentary."
)

# Deterministic decoding, same policy as every other Part A model. repetition_penalty is set
# MILD (1.05, not 1.3) -- the earlier transformers run showed a strong penalty can push the
# model into hallucinated commentary instead of just breaking repetition loops. vLLM doesn't
# support no_repeat_ngram_size (that's a transformers-specific parameter), so repetition_penalty
# is the available lever here; kept mild deliberately.
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=128,
    repetition_penalty=1.05,
)

print("vLLM engine loaded.")

## Cell 6 -- Helper functions: image encoding + conversation builder

In [ ]:
import base64

def image_to_data_url(img_path):
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

def build_conversation(img_path):
    return [
        {"role": "system", "content": OCR_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_to_data_url(img_path)}},
            ],
        },
    ]

## Cell 7 -- Single-image sanity test (run BEFORE the full loop)

In [ ]:
test_row = eval_rows[0]
test_img_path = os.path.join(IMAGES_DIR, image_filename(test_row))

test_outputs = llm.chat([build_conversation(test_img_path)], sampling_params=sampling_params)
test_pred = test_outputs[0].outputs[0].text.strip()

print("file_id:", test_row["file_id"])
print("GROUND TRUTH:", test_row["text"])
print("PREDICTION:  ", test_pred if test_pred else "[EMPTY]")
print("\nCheck: real Urdu text (even if imperfect)? No repetition loops, no meta-commentary,")
print("no refusal/hedging? If wrong, stop and check the config before running the full loop.")

## Cell 8 -- Timed mini-batch (confirm real speedup before committing to the full run)
Runs 20 images through a single batched `llm.chat()` call and reports sec/image, so we can confirm this is actually fast before running all 2,465.

In [ ]:
import time

# NOTE: 20 images previously OOM'd on the T4 here even after fixing bitsandbytes -- the vision
# encoder's activation memory for a large image batch isn't covered by gpu_memory_utilization's
# KV-cache budget. Reduced to 12 as a safer probe; combined with the gpu_memory_utilization and
# max_num_batched_tokens changes in Cell 5, this should now fit comfortably.
timing_rows = eval_rows[:12]
conversations = [build_conversation(os.path.join(IMAGES_DIR, image_filename(r))) for r in timing_rows]

start = time.time()
batch_outputs = llm.chat(conversations, sampling_params=sampling_params)
elapsed = time.time() - start

print(f"Processed {len(timing_rows)} images in {elapsed:.1f}s ({elapsed/len(timing_rows):.2f} sec/image)")
print(f"Projected time for full 2,465 images: {elapsed/len(timing_rows)*2465/60:.1f} minutes")
print("\nCompare this to the transformers version's 23 sec/image -- if this isn't dramatically")
print("faster, stop and report back before running the full loop.")

## Cell 9 -- Run over the evaluation set in batches, with checkpointing (safe to interrupt and resume)
Set `PILOT_SIZE = 100` first if you want another sanity check at scale, or `None` to go straight to the full 2,465 (Cell 7 + Cell 8 already validated correctness and speed).

In [ ]:
PILOT_SIZE = None   # set to 100 for another intermediate check, or None for the full run
BATCH_SIZE = 16      # images per llm.chat() call -- lowered from 50 after a 20-image batch OOM'd
                      # on the T4's vision-encoder memory; re-check Cell 8's timing before raising this
OUTPUT_CSV = "/kaggle/working/qwen3vl_vllm_predictions.csv"

rows_to_run = eval_rows if PILOT_SIZE is None else eval_rows[:PILOT_SIZE]

done_ids = set()
if os.path.isfile(OUTPUT_CSV):
    with open(OUTPUT_CSV, encoding="utf-8") as f:
        done_ids = {r["file_id"] for r in csv.DictReader(f)}
    print(f"Resuming -- {len(done_ids)} already done, skipping those")

remaining_rows = [r for r in rows_to_run if r["file_id"] not in done_ids]
print(f"Rows to process this run: {len(remaining_rows)}")

fieldnames = ["sentence_id", "file_id", "writer_id", "ground_truth", "prediction",
              "cer_sensitive", "cer_insensitive", "wer_sensitive", "wer_insensitive", "error"]

write_header = not os.path.isfile(OUTPUT_CSV)
out_f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=fieldnames)
if write_header:
    writer.writeheader()

for batch_start in range(0, len(remaining_rows), BATCH_SIZE):
    batch_rows = remaining_rows[batch_start:batch_start + BATCH_SIZE]
    batch_paths = [os.path.join(IMAGES_DIR, image_filename(r)) for r in batch_rows]

    try:
        batch_conversations = [build_conversation(p) for p in batch_paths]
        batch_outputs = llm.chat(batch_conversations, sampling_params=sampling_params)
        predictions = [o.outputs[0].text.strip() for o in batch_outputs]
        batch_errors = [""] * len(batch_rows)
    except Exception as e:
        # whole-batch failure (rare) -- record the error against every row in this batch
        # rather than losing the batch silently
        predictions = [""] * len(batch_rows)
        batch_errors = [str(e)] * len(batch_rows)

    for row, prediction, error_msg in zip(batch_rows, predictions, batch_errors):
        if error_msg:
            cer_s = cer_i = wer_s = wer_i = 1.0
        else:
            cer_s, cer_i, wer_s, wer_i = compute_metrics(prediction, row["text"])

        writer.writerow({
            "sentence_id": row["sentence_id"],
            "file_id": row["file_id"],
            "writer_id": row["writer_id"],
            "ground_truth": row["text"],
            "prediction": prediction,
            "cer_sensitive": cer_s,
            "cer_insensitive": cer_i,
            "wer_sensitive": wer_s,
            "wer_insensitive": wer_i,
            "error": error_msg,
        })

    out_f.flush()
    print(f"Batch {batch_start // BATCH_SIZE + 1} / {(len(remaining_rows) + BATCH_SIZE - 1) // BATCH_SIZE} done "
          f"({batch_start + len(batch_rows)} / {len(remaining_rows)} images)")

out_f.close()
print(f"\nDone. Predictions saved to {OUTPUT_CSV}")

## Cell 10 -- Aggregate results

In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT_CSV)
results["error"] = results["error"].fillna("")
results["prediction"] = results["prediction"].fillna("")
print(f"Total predictions: {len(results)}")
print(f"Failed/errored images: {(results['error'] != '').sum()}")

print("\n--- AGGREGATE METRICS (mean across all sentences) ---")
print(f"CER (diacritic-sensitive):   {results['cer_sensitive'].mean():.4f}")
print(f"CER (diacritic-insensitive): {results['cer_insensitive'].mean():.4f}")
print(f"WER (diacritic-sensitive):   {results['wer_sensitive'].mean():.4f}")
print(f"WER (diacritic-insensitive): {results['wer_insensitive'].mean():.4f}")

print(f"\nSentences with 0% CER (perfect match): {(results['cer_sensitive']==0).sum()} / {len(results)}")
print(f"Sentences with 100%+ CER (worse than blank): {(results['cer_sensitive']>=1.0).sum()} / {len(results)}")
print(f"Empty predictions: {(results['prediction']=='').sum()} / {len(results)}")

## Cell 11 -- Error analysis: actual sentence vs prediction, worst/best cases, diagnosis

In [ ]:
def show_examples(df, title, n=5):
    print(f"\n{'='*80}\n{title}\n{'='*80}")
    for _, row in df.head(n).iterrows():
        print(f"\nfile_id: {row['file_id']}  |  CER: {row['cer_sensitive']:.2f}  |  WER: {row['wer_sensitive']:.2f}")
        print(f"  GROUND TRUTH: {row['ground_truth']}")
        pred_display = row['prediction'] if row['prediction'] else '[EMPTY]'
        print(f"  PREDICTION:   {pred_display}")

worst = results.sort_values("cer_sensitive", ascending=False)
best = results.sort_values("cer_sensitive", ascending=True)

show_examples(worst, "WORST 5 PREDICTIONS (highest CER)")
show_examples(best[best["cer_sensitive"] > 0], "BEST NON-PERFECT 5 PREDICTIONS", n=5)

print(f"\n{'='*80}\nDIAGNOSIS\n{'='*80}")
empty_pred = (results["prediction"] == "")
print(f"- {empty_pred.sum()} / {len(results)} predictions were completely empty.")

diacritic_only_error = ((results["cer_sensitive"] > 0) & (results["cer_insensitive"] < results["cer_sensitive"] * 0.5))
print(f"- {diacritic_only_error.sum()} / {len(results)} sentences see CER drop by >50% once diacritics are ignored.")

near_total_failure = (results["cer_sensitive"] >= 0.9)
print(f"- {near_total_failure.sum()} / {len(results)} sentences have CER >= 0.9 (near-total failure).")

long_preds = results[results["prediction"].str.len() > results["ground_truth"].str.len() * 2]
print(f"- {len(long_preds)} / {len(results)} predictions are over 2x longer than the ground truth")
print("  (check these manually for repetition loops or meta-commentary leaking through.)")

## Cell 12 -- Save a clean summary report

In [ ]:
summary_path = "/kaggle/working/qwen3vl_vllm_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("QWEN3-VL-8B-INSTRUCT (vLLM) BASELINE -- RESULTS SUMMARY\n")
    f.write("="*50 + "\n")
    f.write(f"Images evaluated: {len(results)}\n")
    f.write(f"CER (diacritic-sensitive):   {results['cer_sensitive'].mean():.4f}\n")
    f.write(f"CER (diacritic-insensitive): {results['cer_insensitive'].mean():.4f}\n")
    f.write(f"WER (diacritic-sensitive):   {results['wer_sensitive'].mean():.4f}\n")
    f.write(f"WER (diacritic-insensitive): {results['wer_insensitive'].mean():.4f}\n")
    f.write(f"Empty predictions: {(results['prediction']=='').sum()} / {len(results)}\n")

print(f"Saved: {summary_path}")
print(f"Predictions CSV: {OUTPUT_CSV}")